# Gaussian: noise-aware CE from an assumed label-flip rate

## 1. Question and comparison rule

Compare ordinary CE with **forward noise-aware CE**, derived from independent symmetric binary label flips.

- **eta:** actual probability used to flip training labels.
- **eta_expected:** assumed flip probability used inside the noise-aware likelihood.

The model's sigmoid output estimates the **clean** class-1 probability. Transform it to the observed-label probability only during noise-aware training. Validation and test labels remain clean, and their ordinary CE is computed from the untransformed clean probability.

Sweep eta and eta_expected, select learning rate separately for every pair using clean validation CE over three tuning seeds, then evaluate on five new matched seeds. The CE baseline is separately validation-tuned at each actual eta. Data, initializations, corrupted labels, optimizer, and epoch budgets are matched.

Each cell shows
$$
D(\eta,\eta_{\mathrm{expected}})
=\overline{\mathrm{CE}_{\mathrm{test}}(\mathrm{CE})-
\mathrm{CE}_{\mathrm{test}}(\mathrm{NoiseAware})}.
$$
Positive/green means noise-aware training has lower clean test CE; negative/red means ordinary CE is better; zero or small absolute differences are white.

The notebook runs independently on CPU with uv. It reuses compatible CE-baseline records from notebook 03 when available; otherwise it trains them locally. All plots use Plotly and no tables are displayed.


In [1]:
import ast
import csv
import hashlib
import json
import math
import statistics
import time
from itertools import product
from pathlib import Path

import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "pyproject.toml").exists()
RESULTS_ROOT = PROJECT_ROOT / "results" / "06_gaussian_noise_aware"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cpu")
DTYPE = torch.float32
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
pio.templates.default = "plotly_white"
pio.renderers.default = "plotly_mimetype"

ETAS = (0.0, 0.025, 0.05, 0.1, 0.2, 0.3, 0.4)
EXPECTED_ETAS = (0.0, 0.01, 0.025, 0.05, 0.075, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.45)
LEARNING_RATES = (0.001, 0.01, 0.05)
TUNING_SEEDS = (0, 1, 2)
EVALUATION_SEEDS = (201, 202, 203, 204, 205)
SPLIT_SIZES = {"train": 1000, "validation": 400, "test": 2000}
EPOCHS = 400
WIN_ATOL = 1e-6
REUSE_CACHE = True
REUSE_CE_BASELINE = True

assert set(TUNING_SEEDS).isdisjoint(EVALUATION_SEEDS)
assert ETAS == tuple(sorted(set(ETAS))) and EXPECTED_ETAS == tuple(sorted(set(EXPECTED_ETAS)))
assert all(0 <= eta < 0.5 for eta in ETAS)
assert all(0 <= eta_expected < 0.5 for eta_expected in EXPECTED_ETAS) and 0.0 in EXPECTED_ETAS
assert all(size % 2 == 0 for size in SPLIT_SIZES.values())
TOTAL_TUNING_RUNS = len(ETAS) * len(EXPECTED_ETAS) * len(LEARNING_RATES) * len(TUNING_SEEDS)
TOTAL_EVALUATION_RUNS = len(ETAS) * len(EXPECTED_ETAS) * len(EVALUATION_SEEDS)

print(f"PyTorch {torch.__version__} | {DEVICE} | {DTYPE}")
print(f"Grid: {len(ETAS)} eta values x {len(EXPECTED_ETAS)} eta_expected values.")
print(f"Tuning: {TOTAL_TUNING_RUNS} unique runs; evaluation: {TOTAL_EVALUATION_RUNS} unique runs.")
print(f"NoiseAware win: mean CE-minus-NoiseAware difference > {WIN_ATOL:g}.")


PyTorch 2.11.0+cpu | cpu | torch.float32
Grid: 7 eta values x 12 eta_expected values.
Tuning: 756 unique runs; evaluation: 420 unique runs.
NoiseAware win: mean CE-minus-NoiseAware difference > 1e-06.


## 2. Derivation from the expected flip probability

Let $p=\sigma(z)=P(y=1\mid x)$ describe the clean class, and let $\widetilde y$ be its observed, possibly flipped label. Assume
$$
P(\widetilde y=1\mid y=0,x)=P(\widetilde y=0\mid y=1,x)=\alpha,\qquad \alpha=\eta_{\mathrm{expected}}.
$$
Marginalizing the unobserved clean label gives
$$
q=P(\widetilde y=1\mid x)
=(1-\alpha)p+\alpha(1-p)
=\alpha+(1-2\alpha)p.
$$
The noise-aware negative log likelihood of the observed training label is
$$
L_{\mathrm{NA}}(z,\widetilde y;\alpha)
=-\widetilde y\log q-(1-\widetilde y)\log(1-q).
$$

For observed signed margin $m=(2\widetilde y-1)z$, an equivalent stable form is
$$
L_{\mathrm{NA}}
=-\log\left[(1-\alpha)\sigma(m)+\alpha\sigma(-m)\right].
$$
Compute the two log-probability terms using log-sigmoid and combine them with logaddexp. No probability clipping or inversion of corrupted labels is needed.

At alpha = 0 the objective is exactly CE. At alpha = 0.5, q = 0.5 regardless of z, so the observed labels provide no information; the sweep excludes that boundary. Alpha above 0.5 is outside this experiment.

When the assumed noise matches the data-generation rate, this is the likelihood of the specified corruption process. It does not guarantee better finite-sample test CE or eliminate optimization effects. Off-diagonal cells deliberately test misspecified expected noise.


## 3. Matched data and training

Use the same balanced Gaussian classes as notebook 03:
$$
x\mid y\sim\mathcal N((2y-1)(1,1)^\top,I_2),\qquad y\in\{0,1\}.
$$
Generate independent train, validation, and test splits; fit feature standardization only on training features. The logistic model has 3 parameters and trains for 400 full-batch Adam steps.

Each seed varies the data and initial parameters. Reuse both across all actual/expected noise rates and learning rates. For each seed draw one uniform vector, then flip training label i iff its draw is below actual eta. All expected-noise assumptions share exactly the same observed labels for a given seed and actual eta.

The observed targets stay binary; expected eta changes the training likelihood, not the targets. Only training labels are corrupted. Evaluate clean validation/test CE using the model's clean logits z, not the noisy observation probability q.

Expected eta = 0 is the ordinary-CE baseline. Reuse it once per actual eta/seed/lr combination.


In [2]:
def generator(seed):
    return torch.Generator(device="cpu").manual_seed(seed)


def gaussian_split(n, seed):
    g = generator(seed)
    labels = torch.cat((torch.zeros(n // 2), torch.ones(n // 2))).to(DTYPE)
    features = torch.randn(n, 2, generator=g, dtype=DTYPE) + (2 * labels - 1).unsqueeze(1)
    permutation = torch.randperm(n, generator=g)
    return features[permutation].to(DEVICE), labels[permutation].to(DEVICE)


def prepare_data(seed, include_test=False):
    splits = ("train", "validation", "test") if include_test else ("train", "validation")
    data = {
        split: gaussian_split(SPLIT_SIZES[split], seed * 1000 + split_id)
        for split_id, split in enumerate(splits)
    }
    mean = data["train"][0].mean(dim=0)
    scale = data["train"][0].std(dim=0, unbiased=False)
    assert torch.all(scale > 0)
    for split, (features, labels) in data.items():
        data[split] = ((features - mean) / scale, labels)
    data["corruption_uniforms"] = torch.rand(
        SPLIT_SIZES["train"], generator=generator(seed * 1000 + 999), dtype=DTYPE
    )
    return data


def observed_labels(data, eta):
    clean = data["train"][1]
    observed = torch.where(data["corruption_uniforms"] < eta, 1 - clean, clean)
    realized = (observed != clean).to(DTYPE).mean().item()
    return observed, realized


def make_model(seed):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed + 10_000)
        model = nn.Linear(2, 1)
    return model.to(device=DEVICE, dtype=DTYPE)


assert sum(p.numel() for p in make_model(0).parameters()) == 3


In [3]:
def objective(logits, labels, eta_expected):
    ce = F.softplus(logits) - labels * logits
    if eta_expected == 0:
        return ce.mean()
    if not 0 < eta_expected < 0.5:
        raise ValueError("eta_expected must be in [0, 0.5).")
    observed_margin = (2 * labels - 1) * logits
    log_unflipped = math.log1p(-eta_expected) + F.logsigmoid(observed_margin)
    log_flipped = math.log(eta_expected) + F.logsigmoid(-observed_margin)
    return -torch.logaddexp(log_unflipped, log_flipped).mean()


@torch.no_grad()
def evaluate(model, split):
    features, labels = split
    logits = model(features).squeeze(-1)
    if not torch.isfinite(logits).all():
        raise FloatingPointError("Nonfinite predictions.")
    return {
        "ce": F.binary_cross_entropy_with_logits(logits, labels).item(),
        "accuracy": ((logits >= 0) == labels.bool()).to(DTYPE).mean().item(),
        "confidence": torch.sigmoid(logits.abs()).mean().item(),
    }


def train_one(seed, data, observed, eta_expected, lr):
    model = make_model(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    features = data["train"][0]
    started = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        optimizer.zero_grad(set_to_none=True)
        loss = objective(model(features).squeeze(-1), observed, eta_expected)
        if epoch % 20 == 0 and not torch.isfinite(loss):
            raise FloatingPointError(f"Nonfinite loss at epoch {epoch}.")
        loss.backward()
        optimizer.step()
    seconds = time.perf_counter() - started
    model.eval()
    return model, evaluate(model, data["validation"]), seconds


def summarize(values):
    return statistics.mean(values), statistics.stdev(values) if len(values) > 1 else 0.0


def write_csv(path, records):
    if not records:
        return
    fields = list(dict.fromkeys(key for row in records for key in row))
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(records)


def read_csv(path):
    records = []
    with path.open(encoding="utf-8", newline="") as file:
        for row in csv.DictReader(file):
            parsed = {}
            for key, value in row.items():
                if key in ("status", "error"):
                    parsed[key] = value
                elif value == "":
                    parsed[key] = None
                elif key == "seed":
                    parsed[key] = int(value)
                else:
                    parsed[key] = float(value)
            records.append(parsed)
    return records


def run_key(record):
    return record["seed"], record["eta"], record["eta_expected"], record["lr"]


### Reproducibility and cache

Cache successful runs under a fingerprint of numerical settings, PyTorch version, and exact data/training sources. Plot-only changes reuse existing runs.

Optionally reuse only the ordinary-CE records from notebook 03. Before import, verify numerical settings, data source, train/evaluation function structure, the zero-expected-noise CE branch, and source fingerprints. Missing/incompatible baselines are trained locally.

Failed runs are retried. A tuning candidate needs all three successful tuning seeds; a heatmap cell needs all five paired evaluation seeds. Incomplete cells are left blank.


In [4]:
settings = {
    "torch_version": str(torch.__version__),
    "device": str(DEVICE), "dtype": str(DTYPE), "threads": torch.get_num_threads(),
    "etas": list(ETAS), "expected_etas": list(EXPECTED_ETAS), "learning_rates": list(LEARNING_RATES),
    "tuning_seeds": list(TUNING_SEEDS), "evaluation_seeds": list(EVALUATION_SEEDS),
    "split_sizes": SPLIT_SIZES, "epochs": EPOCHS, "win_atol": WIN_ATOL,
    "optimizer": "Adam, default betas/eps, no weight decay, no schedule",
    "batching": "full batch", "selection": "mean final ordinary clean validation CE",
}
notebook_path = PROJECT_ROOT / "notebooks" / "06_gaussian_noise_aware.ipynb"
notebook_source = json.loads(notebook_path.read_text(encoding="utf-8"))
implementation = "\n".join(
    "".join(cell["source"]) for cell in notebook_source["cells"]
    if cell.get("id") in ("data", "training")
)
fingerprint = hashlib.sha256(
    json.dumps({"settings": settings, "implementation": implementation}, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
RUN_DIR = RESULTS_ROOT / fingerprint
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "settings.json").write_text(json.dumps(settings, indent=2) + "\n", encoding="utf-8")
(RESULTS_ROOT / "latest_run.json").write_text(
    json.dumps({"run_id": fingerprint, "run_directory": str(RUN_DIR)}, indent=2) + "\n", encoding="utf-8"
)
TUNING_PATH = RUN_DIR / "tuning_runs.csv"
EVALUATION_PATH = RUN_DIR / "evaluation_runs.csv"
tuning_cache = {
    run_key(row): row for row in (read_csv(TUNING_PATH) if REUSE_CACHE and TUNING_PATH.exists() else [])
}
evaluation_cache = {
    run_key(row): row for row in (read_csv(EVALUATION_PATH) if REUSE_CACHE and EVALUATION_PATH.exists() else [])
}
print(f"Result directory: {RUN_DIR}")
print(f"Cached records: {len(tuning_cache)} tuning, {len(evaluation_cache)} evaluation.")

# Optional CE-only reuse from notebook 03, guarded by settings and source checks.
def canonical_function(source, name, first_statements=None):
    module = ast.parse(source)
    node = next(
        item for item in module.body
        if isinstance(item, (ast.FunctionDef, ast.AsyncFunctionDef)) and item.name == name
    )

    class NormalizeStrength(ast.NodeTransformer):
        def visit_Name(self, item):
            if item.id in ("delta", "eta_expected"):
                item.id = "strength"
            return item

        def visit_arg(self, item):
            if item.arg in ("delta", "eta_expected"):
                item.arg = "strength"
            return item

    node = NormalizeStrength().visit(node)
    if first_statements is not None:
        node.body = node.body[:first_statements]
    return ast.dump(node, include_attributes=False)


def compatible_ce_source():
    latest_path = PROJECT_ROOT / "results" / "03_gaussian_eta_delta" / "latest_run.json"
    source_path = PROJECT_ROOT / "notebooks" / "03_gaussian_eta_delta.ipynb"
    if not latest_path.exists() or not source_path.exists():
        return None
    latest = json.loads(latest_path.read_text(encoding="utf-8"))
    directory = Path(latest["run_directory"])
    settings_path = directory / "settings.json"
    if not settings_path.exists():
        return None
    base_settings = json.loads(settings_path.read_text(encoding="utf-8"))
    common_keys = (
        "torch_version", "device", "dtype", "threads", "etas", "learning_rates",
        "tuning_seeds", "evaluation_seeds", "split_sizes", "epochs", "win_atol",
        "optimizer", "batching", "selection",
    )
    if any(base_settings.get(key) != settings[key] for key in common_keys):
        return None
    baseline_notebook = json.loads(source_path.read_text(encoding="utf-8"))
    baseline_cells = {
        cell.get("id"): "".join(cell["source"]) for cell in baseline_notebook["cells"]
    }
    current_cells = {
        cell.get("id"): "".join(cell["source"]) for cell in notebook_source["cells"]
    }
    baseline_implementation = "\n".join(
        "".join(cell["source"]) for cell in baseline_notebook["cells"]
        if cell.get("id") in ("data", "training")
    )
    baseline_fingerprint = hashlib.sha256(
        json.dumps(
            {"settings": base_settings, "implementation": baseline_implementation},
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()[:16]
    if baseline_fingerprint != latest.get("run_id"):
        return None
    if baseline_cells["data"] != current_cells["data"]:
        return None
    for function in ("train_one", "evaluate"):
        if canonical_function(baseline_cells["training"], function) != canonical_function(current_cells["training"], function):
            return None
    if canonical_function(baseline_cells["training"], "objective", 2) != canonical_function(current_cells["training"], "objective", 2):
        return None
    return directory


baseline_source = compatible_ce_source() if REUSE_CACHE and REUSE_CE_BASELINE else None
imported_baseline_tuning = 0
imported_baseline_evaluation = 0
if baseline_source is not None:
    for filename, cache, allowed_seeds, phase in (
        ("tuning_runs.csv", tuning_cache, TUNING_SEEDS, "tuning"),
        ("evaluation_runs.csv", evaluation_cache, EVALUATION_SEEDS, "evaluation"),
    ):
        source_path = baseline_source / filename
        if not source_path.exists():
            continue
        for row in read_csv(source_path):
            if row.get("delta") != 0 or row["status"] != "ok":
                continue
            if row["seed"] not in allowed_seeds or row["eta"] not in ETAS or row["lr"] not in LEARNING_RATES:
                continue
            row = dict(row)
            row["eta_expected"] = row.pop("delta")
            key = run_key(row)
            if key in cache and cache[key]["status"] == "ok":
                continue
            cache[key] = row
            if phase == "tuning":
                imported_baseline_tuning += 1
            else:
                imported_baseline_evaluation += 1
    write_csv(TUNING_PATH, list(tuning_cache.values()))
    write_csv(EVALUATION_PATH, list(evaluation_cache.values()))
    print(
        f"Imported compatible CE baselines: {imported_baseline_tuning} tuning, "
        f"{imported_baseline_evaluation} evaluation records."
    )
else:
    print("No compatible external CE baseline imported; missing baselines will be trained locally.")

reuse_manifest_path = RUN_DIR / "baseline_reuse.json"
if not reuse_manifest_path.exists():
    reuse_manifest_path.write_text(
        json.dumps({
            "source_directory": str(baseline_source) if baseline_source is not None else None,
            "imported_tuning_records": imported_baseline_tuning,
            "imported_evaluation_records": imported_baseline_evaluation,
        }, indent=2) + "\n",
        encoding="utf-8",
    )


Result directory: C:\Users\shich\Src\hw1opt\results\06_gaussian_noise_aware\4f5840cc4e95d6ec
Cached records: 756 tuning, 420 evaluation.
Imported compatible CE baselines: 0 tuning, 0 evaluation records.


## 4. Select learning rate using clean validation CE

For each actual eta and expected eta, search the same three learning rates on three tuning seeds. Corrupt training labels using actual eta, while the objective uses eta_expected. Select by mean final ordinary CE on clean validation labels, computed from clean logits.

Ordinary CE is expected eta = 0 and receives its own learning-rate selection at every actual eta. No test split is generated during tuning. Expected eta itself is not selected using test results.


In [5]:
started = time.perf_counter()
new_tuning_runs = 0
tuning_data = {seed: prepare_data(seed, include_test=False) for seed in TUNING_SEEDS}

for seed, eta in product(TUNING_SEEDS, ETAS):
    data = tuning_data[seed]
    observed, realized = observed_labels(data, eta)
    for eta_expected, lr in product(EXPECTED_ETAS, LEARNING_RATES):
        record = dict(seed=seed, eta=eta, eta_expected=eta_expected, lr=lr, realized_corruption=realized)
        key = run_key(record)
        if key in tuning_cache and tuning_cache[key]["status"] == "ok":
            continue
        try:
            _, validation, seconds = train_one(seed, data, observed, eta_expected, lr)
            record.update(
                status="ok", validation_ce=validation["ce"],
                validation_accuracy=validation["accuracy"], training_seconds=seconds,
            )
        except (FloatingPointError, RuntimeError) as error:
            record.update(status="failed", error=str(error))
        tuning_cache[key] = record
        new_tuning_runs += 1
    write_csv(TUNING_PATH, list(tuning_cache.values()))
    print(f"Tuning seed={seed}, eta={eta:g}: {len(tuning_cache)}/{TOTAL_TUNING_RUNS} records.", flush=True)

tuning_records = list(tuning_cache.values())
tuning_summary = []
selected = {}
for eta, eta_expected in product(ETAS, EXPECTED_ETAS):
    candidates = []
    for lr in LEARNING_RATES:
        runs = [tuning_cache.get((seed, eta, eta_expected, lr)) for seed in TUNING_SEEDS]
        eligible = all(row is not None and row["status"] == "ok" for row in runs)
        entry = dict(eta=eta, eta_expected=eta_expected, lr=lr, eligible=eligible)
        if eligible:
            entry["validation_ce_mean"], entry["validation_ce_std"] = summarize([r["validation_ce"] for r in runs])
            candidates.append(entry)
        tuning_summary.append(entry)
    if not candidates:
        raise RuntimeError(f"No complete valid learning-rate candidate for eta={eta}, eta_expected={eta_expected}.")
    best = min(candidates, key=lambda row: (row["validation_ce_mean"], row["lr"]))
    selected[(eta, eta_expected)] = best["lr"]

frozen_settings = [
    dict(eta=eta, eta_expected=eta_expected, lr=lr) for (eta, eta_expected), lr in selected.items()
]
selection_snapshot = json.dumps(frozen_settings, sort_keys=True)
(RUN_DIR / "selected_settings.json").write_text(json.dumps(frozen_settings, indent=2) + "\n", encoding="utf-8")
write_csv(RUN_DIR / "tuning_summary.csv", tuning_summary)
boundary_count = sum(lr in (min(LEARNING_RATES), max(LEARNING_RATES)) for lr in selected.values())
print(f"Tuning finished in {time.perf_counter()-started:.1f} s; {new_tuning_runs} newly trained runs.")
print(f"Selected rates at tested grid endpoints: {boundary_count}/{len(selected)}. Values outside this finite grid were not tested.")


Tuning seed=0, eta=0: 756/756 records.


Tuning seed=0, eta=0.025: 756/756 records.


Tuning seed=0, eta=0.05: 756/756 records.


Tuning seed=0, eta=0.1: 756/756 records.


Tuning seed=0, eta=0.2: 756/756 records.


Tuning seed=0, eta=0.3: 756/756 records.


... (12 outputs left out)

Tuning seed=2, eta=0.2: 756/756 records.


Tuning seed=2, eta=0.3: 756/756 records.

Tuning seed=2, eta=0.4: 756/756 records.


Tuning finished in 0.2 s; 0 newly trained runs.
Selected rates at tested grid endpoints: 66/84. Values outside this finite grid were not tested.


## 5. Evaluation on matched seeds

Use evaluation seeds 201 through 205, as in notebooks 03 and 04. Train from scratch using frozen validation-selected learning rates. Each actual/expected-noise cell gets the same data, initial model, and label flips as its CE baseline.

The final model outputs a clean-class probability p. Compute ordinary clean test CE from z and the clean test labels. Do not use the transformed observation probability q for this metric.

Actual eta is the generative flip probability; the realized fraction can differ by seed and is saved. Expected eta is an independently specified modeling assumption. Matched actual/expected values lie on the known-noise diagonal of the grid.


In [6]:
started = time.perf_counter()
new_evaluation_runs = 0
evaluation_data = {seed: prepare_data(seed, include_test=True) for seed in EVALUATION_SEEDS}

for seed, eta in product(EVALUATION_SEEDS, ETAS):
    data = evaluation_data[seed]
    observed, realized = observed_labels(data, eta)
    for eta_expected in EXPECTED_ETAS:
        lr = selected[(eta, eta_expected)]
        record = dict(seed=seed, eta=eta, eta_expected=eta_expected, lr=lr, realized_corruption=realized)
        key = run_key(record)
        if key in evaluation_cache and evaluation_cache[key]["status"] == "ok":
            continue
        try:
            model, validation, seconds = train_one(seed, data, observed, eta_expected, lr)
            test = evaluate(model, data["test"])
            record.update(
                status="ok", validation_ce=validation["ce"],
                test_ce=test["ce"], test_accuracy=test["accuracy"],
                test_confidence=test["confidence"], training_seconds=seconds,
            )
        except (FloatingPointError, RuntimeError) as error:
            record.update(status="failed", error=str(error))
        evaluation_cache[key] = record
        new_evaluation_runs += 1
    write_csv(EVALUATION_PATH, list(evaluation_cache.values()))
    print(f"Evaluation seed={seed}, eta={eta:g}: {len(evaluation_cache)}/{TOTAL_EVALUATION_RUNS} records.", flush=True)

evaluation_records = list(evaluation_cache.values())
assert selection_snapshot == json.dumps([
    dict(eta=eta, eta_expected=eta_expected, lr=lr) for (eta, eta_expected), lr in selected.items()
], sort_keys=True)
print(f"Evaluation finished in {time.perf_counter()-started:.1f} s; {new_evaluation_runs} newly trained runs.")


Evaluation seed=201, eta=0: 420/420 records.


Evaluation seed=201, eta=0.025: 420/420 records.


Evaluation seed=201, eta=0.05: 420/420 records.

Evaluation seed=201, eta=0.1: 420/420 records.


Evaluation seed=201, eta=0.2: 420/420 records.


Evaluation seed=201, eta=0.3: 420/420 records.


... (26 outputs left out)

Evaluation seed=205, eta=0.2: 420/420 records.


Evaluation seed=205, eta=0.3: 420/420 records.


Evaluation seed=205, eta=0.4: 420/420 records.

Evaluation finished in 0.2 s; 0 newly trained runs.


## 6. CE minus NoiseAware

For each cell, subtract the noise-aware model's test loss from the ordinary-CE baseline test loss within the same seed and actual corruption condition. Average these five paired differences.

Mean difference > +0.000001 is an observed NoiseAware win. Mean difference < -0.000001 favors CE. Values within the tolerance are ties. Save sample standard deviation, relative change, and seed-level win counts alongside both test losses.

Expected eta = 0 must be an exact shared-CE control with zero differences. This exploratory test grid does not select an expected-noise setting for deployment.


In [7]:
paired_records = []
cell_records = []
for eta, eta_expected in product(ETAS, EXPECTED_ETAS):
    baseline_rows, noise_aware_rows, differences = [], [], []
    for seed in EVALUATION_SEEDS:
        baseline = evaluation_cache.get((seed, eta, 0.0, selected[(eta, 0.0)]))
        noise_aware = evaluation_cache.get((seed, eta, eta_expected, selected[(eta, eta_expected)]))
        if baseline is None or noise_aware is None or baseline["status"] != "ok" or noise_aware["status"] != "ok":
            continue
        assert baseline["realized_corruption"] == noise_aware["realized_corruption"]
        difference = baseline["test_ce"] - noise_aware["test_ce"]
        if eta_expected == 0:
            assert difference == 0
        baseline_rows.append(baseline)
        noise_aware_rows.append(noise_aware)
        differences.append(difference)
        paired_records.append(dict(
            seed=seed, eta=eta, eta_expected=eta_expected,
            realized_corruption=baseline["realized_corruption"],
            baseline_lr=baseline["lr"], noise_aware_lr=noise_aware["lr"],
            baseline_test_ce=baseline["test_ce"], noise_aware_test_ce=noise_aware["test_ce"],
            difference_ce_minus_noise_aware=difference, gain=difference,
            noise_aware_win=difference > WIN_ATOL, ce_win=difference < -WIN_ATOL,
        ))
    complete = len(differences) == len(EVALUATION_SEEDS)
    row = dict(
        eta=eta, eta_expected=eta_expected, complete=complete, paired_seeds=len(differences),
        expected_matches_actual=math.isclose(eta_expected, eta, rel_tol=0, abs_tol=1e-12),
        baseline_lr=selected[(eta, 0.0)], noise_aware_lr=selected[(eta, eta_expected)],
    )
    if complete:
        mean_difference, std_difference = summarize(differences)
        baseline_mean = statistics.mean(r["test_ce"] for r in baseline_rows)
        noise_aware_mean = statistics.mean(r["test_ce"] for r in noise_aware_rows)
        winner = "NoiseAware" if mean_difference > WIN_ATOL else "CE" if mean_difference < -WIN_ATOL else "tie"
        row.update(
            baseline_test_ce_mean=baseline_mean, noise_aware_test_ce_mean=noise_aware_mean,
            difference_mean=mean_difference, difference_std=std_difference,
            gain_mean=mean_difference, relative_difference_percent=100 * mean_difference / baseline_mean,
            noise_aware_seed_wins=sum(value > WIN_ATOL for value in differences),
            ce_seed_wins=sum(value < -WIN_ATOL for value in differences),
            seed_ties=sum(abs(value) <= WIN_ATOL for value in differences), winner=winner,
            realized_corruption_mean=statistics.mean(r["realized_corruption"] for r in baseline_rows),
            noise_aware_confidence_mean=statistics.mean(r["test_confidence"] for r in noise_aware_rows),
            baseline_confidence_mean=statistics.mean(r["test_confidence"] for r in baseline_rows),
        )
        assert math.isclose(mean_difference, baseline_mean - noise_aware_mean, rel_tol=0, abs_tol=1e-14)
    cell_records.append(row)

write_csv(RUN_DIR / "paired_results.csv", paired_records)
write_csv(RUN_DIR / "grid_summary.csv", cell_records)
complete_cells = [r for r in cell_records if r["complete"]]
nonzero_cells = [r for r in complete_cells if r["eta_expected"] > 0]
winning_cells = [r for r in nonzero_cells if r["winner"] == "NoiseAware"]
winner_counts = {name: sum(r["winner"] == name for r in complete_cells) for name in ("NoiseAware", "CE", "tie")}
print(f"Complete cells: {len(complete_cells)}/{len(ETAS)*len(EXPECTED_ETAS)}.")
print(f"Observed mean winners: NoiseAware={winner_counts['NoiseAware']}, CE={winner_counts['CE']}, ties={winner_counts['tie']}.")
for eta in ETAS:
    diagonal = next((r for r in complete_cells if r["eta"] == eta and r["expected_matches_actual"]), None)
    if diagonal is not None:
        print(
            f"Matched eta_expected=eta={eta:g}: CE={diagonal['baseline_test_ce_mean']:.5f}, "
            f"NoiseAware={diagonal['noise_aware_test_ce_mean']:.5f}, "
            f"difference={diagonal['difference_mean']:+.5f}."
        )


Complete cells: 84/84.


Observed mean winners: NoiseAware=63, CE=14, ties=7.
Matched eta_expected=eta=0: CE=0.19209, NoiseAware=0.19209, difference=+0.00000.
Matched eta_expected=eta=0.025: CE=0.20282, NoiseAware=0.19215, difference=+0.01067.
Matched eta_expected=eta=0.05: CE=0.22175, NoiseAware=0.19262, difference=+0.02913.
Matched eta_expected=eta=0.1: CE=0.26635, NoiseAware=0.19279, difference=+0.07356.
Matched eta_expected=eta=0.2: CE=0.36119, NoiseAware=0.19326, difference=+0.16793.
Matched eta_expected=eta=0.3: CE=0.46334, NoiseAware=0.20789, difference=+0.25545.
Matched eta_expected=eta=0.4: CE=0.56747, NoiseAware=0.25301, difference=+0.31446.


## 7. Expected versus actual noise: red-white-green grid

Every cell shows the five-seed mean **ordinary clean test CE(CE baseline) - CE(NoiseAware)**.

- **Green / positive:** noise-aware training improves clean test CE.
- **Red / negative:** ordinary CE is better.
- **White:** zero or small absolute difference.
- **Saturation:** magnitude of the difference on one symmetric linear scale.
- **Missing:** fewer than five valid paired evaluation seeds.

Horizontal bins are expected flip probabilities; vertical bins are actual flip probabilities. The expected-zero column is a white CE identity control. The cells where expected eta equals actual eta test a correctly specified noise assumption.

Hover shows both losses, paired variability, seed wins, learning rates, realized corruption, and whether expected/actual noise match.


In [8]:
by_cell = {(r["eta"], r["eta_expected"]): r for r in cell_records}
color_grid, text_grid, hover_grid = [], [], []
for eta in ETAS:
    colors, texts, details = [], [], []
    for eta_expected in EXPECTED_ETAS:
        row = by_cell[(eta, eta_expected)]
        if not row["complete"]:
            colors.append(None)
            texts.append("missing")
            details.append([eta, eta_expected] + [None] * 13)
            continue
        difference = row["difference_mean"]
        colors.append(0.0 if abs(difference) <= WIN_ATOL else difference)
        texts.append(
            "tie" if abs(difference) <= WIN_ATOL
            else f"{difference:+.3f}" if abs(difference) >= 0.001
            else f"{difference:+.1e}"
        )
        details.append([
            eta, eta_expected, row["baseline_test_ce_mean"], row["noise_aware_test_ce_mean"],
            difference, row["relative_difference_percent"], row["difference_std"],
            row["noise_aware_seed_wins"], row["ce_seed_wins"], row["paired_seeds"],
            row["baseline_lr"], row["noise_aware_lr"], row["expected_matches_actual"],
            row["realized_corruption_mean"], row["winner"],
        ])
    color_grid.append(colors)
    text_grid.append(texts)
    hover_grid.append(details)

color_limit = max([abs(value) for row in color_grid for value in row if value is not None] + [WIN_ATOL])
fig = go.Figure(go.Heatmap(
    x=[f"{value:g}" for value in EXPECTED_ETAS],
    y=[f"{eta:g}" for eta in ETAS],
    z=color_grid, zmin=-color_limit, zmax=color_limit,
    colorscale=[[0.0, "#991b1b"], [0.5, "#ffffff"], [1.0, "#166534"]],
    text=text_grid, texttemplate="%{text}", textfont=dict(size=10, color="#111827"),
    customdata=hover_grid, xgap=2, ygap=2, hoverongaps=False,
    colorbar=dict(title="CE - CE(NoiseAware)", thickness=18),
    hovertemplate=(
        "Actual eta=%{customdata[0]}<br>Expected eta=%{customdata[1]}"
        "<br>Baseline clean test CE=%{customdata[2]:.8f}"
        "<br>NoiseAware clean test CE=%{customdata[3]:.8f}"
        "<br>CE - NoiseAware=%{customdata[4]:+.8f}"
        "<br>Relative CE reduction=%{customdata[5]:+.4f}%"
        "<br>Paired sample SD=%{customdata[6]:.8f}"
        "<br>Seed wins: NoiseAware=%{customdata[7]}, CE=%{customdata[8]}, paired=%{customdata[9]}"
        "<br>Selected lr: baseline=%{customdata[10]}, NoiseAware=%{customdata[11]}"
        "<br>Expected matches actual=%{customdata[12]}"
        "<br>Mean realized corruption=%{customdata[13]:.4f}"
        "<br>Mean winner=%{customdata[14]}<extra></extra>"
    ),
))
fig.update_xaxes(title_text="Expected label-flip probability eta_expected", type="category", tickangle=-35)
fig.update_yaxes(title_text="Actual training-label flip probability eta", type="category", autorange="reversed")
fig.update_layout(
    title="Gaussian: CE - NoiseAware<br><sup>Ordinary clean test CE; forward symmetric-noise likelihood</sup>",
    height=690, margin=dict(l=90, r=205, t=120, b=120), plot_bgcolor="#e5e7eb",
    annotations=[dict(
        x=0, y=1.08, xref="paper", yref="paper", showarrow=False, xanchor="left",
        text="Green: NoiseAware better; red: CE better; white: near zero. Five matched evaluation seeds.",
    )],
)
fig.show(config={"displaylogo": False, "responsive": True})
fig.write_html(RUN_DIR / "eta_expected_vs_eta_heatmap.html", include_plotlyjs=True, full_html=True)


## 8. Interpretation and scope

Compare expected and actual flip rates. Matching them specifies the correct noise likelihood for the assumed symmetric corruption process; underestimating noise leaves part of the corruption uncorrected, while overestimating can distort clean probabilities. Finite data, optimization, and validation selection can change which cell has the lowest observed test CE.

The evaluated probability is always the model's clean p, never the transformed noisy q. Improving the likelihood of corrupted observations alone would not establish lower clean test CE.

This experiment uses one known synthetic noise mechanism, a correctly specified Gaussian logistic model, fixed sample sizes, 400 optimization steps, and five evaluation seeds. Expected eta is treated as a supplied assumption; it is not estimated by looking at test outcomes. The grid is exploratory and observed wins are not significance claims. A proposed setting should be frozen and checked on fresh seeds before confirmation.


In [9]:
failures = [
    {"phase": phase, **record}
    for phase, records in (("tuning", tuning_records), ("evaluation", evaluation_records))
    for record in records if record["status"] != "ok"
]
protocol = {
    **settings, "run_id": fingerprint,
    "test_metric": "ordinary CE from clean model logits on clean held-out labels",
    "difference": "baseline ordinary clean test CE minus NoiseAware ordinary clean test CE",
    "baseline": "eta_expected=0, independently validation-tuned lr at each actual eta",
    "noise_aware_definition": "q=eta_expected+(1-2*eta_expected)*sigmoid(z); NLL of observed training label",
    "loss_evaluation": "stable logaddexp of unflipped/flipped log-sigmoid likelihoods",
    "assumed_noise_domain": "0 <= eta_expected < 0.5",
    "win_rule": f"five-seed mean CE-minus-NoiseAware difference > {WIN_ATOL}",
    "heatmap_colors": "positive green; negative red; zero-centered white",
    "zero_expected_noise_control": "exact same CE model/record reused; difference must be zero",
    "cache_reuse": REUSE_CACHE,
    "ce_baseline_source": str(baseline_source) if baseline_source is not None else None,
    "unique_tuning_runs": TOTAL_TUNING_RUNS, "unique_evaluation_runs": TOTAL_EVALUATION_RUNS,
    "complete_cells": len(complete_cells), "mean_winner_counts": winner_counts,
    "positive_expected_noise_cells": len(nonzero_cells),
    "observed_winning_cells": len(winning_cells), "failed_runs": len(failures),
    "scope": "exploratory expected-versus-actual noise grid; no test-based model selection",
}
(RUN_DIR / "protocol.json").write_text(json.dumps(protocol, indent=2) + "\n", encoding="utf-8")
(RUN_DIR / "failures.json").write_text(json.dumps(failures, indent=2) + "\n", encoding="utf-8")
assert len(tuning_records) == TOTAL_TUNING_RUNS
assert len(evaluation_records) == TOTAL_EVALUATION_RUNS
assert len(cell_records) == len(ETAS) * len(EXPECTED_ETAS)
assert len([r for r in evaluation_records if r["eta_expected"] == 0]) == len(ETAS) * len(EVALUATION_SEEDS)
for row in complete_cells:
    assert row["noise_aware_seed_wins"] + row["ce_seed_wins"] + row["seed_ties"] == len(EVALUATION_SEEDS)
print(
    f"Saved {len(cell_records)} cells from {len(tuning_records)} tuning and "
    f"{len(evaluation_records)} evaluation records. Failed runs: {len(failures)}."
)
print(f"Artifacts and interactive HTML: {RUN_DIR}")


Saved 84 cells from 756 tuning and 420 evaluation records. Failed runs: 0.
Artifacts and interactive HTML: C:\Users\shich\Src\hw1opt\results\06_gaussian_noise_aware\4f5840cc4e95d6ec
